In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import unary_union

# 1) 输入/输出路径
input_path = "/Users/frank/opensource/test-data/beijing/beijing.geojson"
output_boundary_path = "/Users/frank/opensource/test-data/beijing/beijing_boundary_geopandas.geojson"

# 2) 读取图层并统一到 WGS84(EPSG:4326)
gdf = gpd.read_file(input_path)
if gdf.empty:
    raise ValueError(f"输入数据为空: {input_path}")

if gdf.crs is None:
    # 若源数据未声明坐标系，这里按常见经纬度数据假设为 WGS84
    gdf = gdf.set_crs(epsg=4326)
else:
    gdf = gdf.to_crs(epsg=4326)

# 3) 合并所有要素，得到整体边界多边形
boundary_geom = unary_union(gdf.geometry)
if boundary_geom.is_empty:
    raise ValueError("边界几何为空，无法输出。")

# 4) 计算经纬度范围（bbox）
minx, miny, maxx, maxy = boundary_geom.bounds
print("经度范围 (lon): [{:.6f}, {:.6f}]".format(minx, maxx))
print("纬度范围 (lat): [{:.6f}, {:.6f}]".format(miny, maxy))

# 5) 输出边界多边形到 GeoJSON
boundary_gdf = gpd.GeoDataFrame({"id": [1]}, geometry=[boundary_geom], crs="EPSG:4326")
boundary_gdf.to_file(output_boundary_path, driver="GeoJSON")
print(f"边界多边形已输出: {output_boundary_path}")

# 6) 图示结果：灰色原始要素 + 红色边界
fig, ax = plt.subplots(figsize=(8, 8))
gdf.plot(ax=ax, facecolor="#d9d9d9", edgecolor="#888888", linewidth=0.4)
boundary_gdf.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
ax.set_title("Beijing Boundary (GeoPandas)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [1]:
from pathlib import Path
import json
import numpy as np
import geopandas as gpd
import pyogrio

# 读取文件路径
base = Path('/Users/frank/opensource/test-data/beijing')
pbf_path = base / 'beijing-260416.osm.pbf'
geojson_path = base / 'beijing.geojson'

if not pbf_path.exists():
    raise FileNotFoundError(f'未找到文件: {pbf_path}')
if not geojson_path.exists():
    raise FileNotFoundError(f'未找到文件: {geojson_path}')

# 1) 输出 PBF 元数据（图层、几何类型、范围、文件大小）
layers = pyogrio.list_layers(pbf_path)
layer_meta = []
all_bounds = []

for layer_name, geom_type in layers:
    info = pyogrio.read_info(pbf_path, layer=layer_name)
    b = info.get('total_bounds')
    if b is not None:
        all_bounds.append(b)
    layer_meta.append({
        'layer': str(layer_name),
        'geometry_type': str(geom_type),
        'feature_count': int(info.get('features', -1)),
        'crs': str(info.get('crs')),
        'bounds': [float(x) for x in b] if b is not None else None,
    })

if not all_bounds:
    raise ValueError('无法从 PBF 图层获取范围信息。')

all_bounds = np.array(all_bounds, dtype=float)
pbf_bounds = [
    float(all_bounds[:, 0].min()),
    float(all_bounds[:, 1].min()),
    float(all_bounds[:, 2].max()),
    float(all_bounds[:, 3].max()),
]

pbf_meta = {
    'file': str(pbf_path),
    'size_bytes': int(pbf_path.stat().st_size),
    'layer_count': int(len(layers)),
    'layers': layer_meta,
    'bounds_wgs84': {
        'min_lon': pbf_bounds[0],
        'min_lat': pbf_bounds[1],
        'max_lon': pbf_bounds[2],
        'max_lat': pbf_bounds[3],
    },
}

print('PBF 元数据:')
print(json.dumps(pbf_meta, ensure_ascii=False, indent=2))

# 2) 读取 beijing.geojson 的范围并做一致性验证
gdf = gpd.read_file(geojson_path)
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
else:
    gdf = gdf.to_crs(epsg=4326)

geo_bounds = [float(x) for x in gdf.total_bounds]  # [min_lon, min_lat, max_lon, max_lat]

# 设置容差：0.01 度（约 1km 量级）
tolerance = 0.01
diff = [abs(a - b) for a, b in zip(pbf_bounds, geo_bounds)]
is_consistent = all(d <= tolerance for d in diff)

print('\nGeoJSON 范围 [min_lon, min_lat, max_lon, max_lat]:')
print(geo_bounds)
print('PBF 范围 [min_lon, min_lat, max_lon, max_lat]:')
print(pbf_bounds)
print('逐项绝对差值:', [round(d, 6) for d in diff])
print(f'范围是否一致（容差 ±{tolerance}°）: {is_consistent}')

PBF 元数据:
{
  "file": "/Users/frank/opensource/test-data/beijing/beijing-260416.osm.pbf",
  "size_bytes": 35655388,
  "layer_count": 5,
  "layers": [
    {
      "layer": "points",
      "geometry_type": "Point",
      "feature_count": -1,
      "crs": "EPSG:4326",
      "bounds": [
        115.415497,
        39.440833000000005,
        117.50950800000001,
        41.063822
      ]
    },
    {
      "layer": "lines",
      "geometry_type": "LineString",
      "feature_count": -1,
      "crs": "EPSG:4326",
      "bounds": [
        115.415497,
        39.440833000000005,
        117.50950800000001,
        41.063822
      ]
    },
    {
      "layer": "multilinestrings",
      "geometry_type": "MultiLineString",
      "feature_count": -1,
      "crs": "EPSG:4326",
      "bounds": [
        115.415497,
        39.440833000000005,
        117.50950800000001,
        41.063822
      ]
    },
    {
      "layer": "multipolygons",
      "geometry_type": "MultiPolygon",
      "feature_count"